# Compare Email and Paid Social

*Data Analysis in Python · Lesson 11 final exercise*

Your media team is setting next year's budget, and they want to know whether the customers Paid Social brings in are worth as much as Email's. Compare the customers both channels brought in from January to June 2024: how many ordered again within a year, and which RFM segment they're in today.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between the parameters cell and the **Check your work** cell. If you're stuck on a step, open its hint under the **Check your work** cell.

1. Read `CUSTOMERS_FILE` and `ORDERS_FILE` into the `customers` and `orders` tables, with `created_at` as dates, and keep the orders before `CUTOFF`.
2. Create the `h1_customers` table: the customers created from `COHORT_START` to before `COHORT_END`.
3. Create `repeat_rates`: for each `acquisition_channel`, the share of the `h1_customers` table's customers who placed another order, of any status, after their first order and within 365 days of it. A cancelled order still shows that the customer came back.
4. Create the `rfm` table: topic 11.3.'s recency, frequency, monetary value, scores and `segment`, for every customer with an order before `CUTOFF` that wasn't cancelled, as of `CUTOFF`.
5. Create the `segment_mix` table: one row for each channel in `CHANNELS` and one column per segment, with the share of the channel's `h1_customers` in each segment, among those in the `rfm` table.
6. Add a text cell for the media team: how the two channels' repeat rates compare, where their customers are today, what could explain the gap besides the channel, and what they should do with it.
7. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `h1_customers` table has 7,545 customers. `repeat_rates` has 9 channels: Email's is about `0.3532` and Paid Social's about `0.1955`. The `rfm` table has 41,776 customers. `segment_mix` has a row for Email and one for Paid Social: 7.6% of Email's customers are Champions, against 0.8% of Paid Social's.

## Parameters

In [ ]:
import numpy as np
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
CUSTOMERS_FILE = DATA_URL + "customers.csv"
ORDERS_FILE = DATA_URL + "orders.csv"
CUTOFF = "2026-01-01"  # The timestamps are in UTC: the orders before it are the orders of 2024 and 2025.
COHORT_START, COHORT_END = "2024-01-01", "2024-07-01"
CHANNELS = ["Email", "Paid Social"]

In [ ]:
customers = pd.read_csv(CUSTOMERS_FILE, parse_dates=["created_at"])
orders = pd.read_csv(ORDERS_FILE, parse_dates=["created_at"])
orders = orders[orders["created_at"] < CUTOFF]
h1_customers = customers[(customers["created_at"] >= COHORT_START) & (customers["created_at"] < COHORT_END)]
len(orders), len(h1_customers)

In [ ]:
h1_orders = orders[orders["customer_id"].isin(h1_customers["customer_id"])]
first_orders = h1_orders.groupby("customer_id")["created_at"].min().rename("first_order")
h1_orders = h1_orders.merge(first_orders, on="customer_id", validate="many_to_one")
again = h1_orders[(h1_orders["created_at"] > h1_orders["first_order"])
                  & (h1_orders["created_at"] <= h1_orders["first_order"] + pd.Timedelta(days=365))]
repeated = h1_customers["customer_id"].isin(again["customer_id"])
repeat_rates = repeated.groupby(h1_customers["acquisition_channel"]).mean()
repeat_rates.sort_values(ascending=False).round(3), repeated.mean()

In [ ]:
valid_orders = orders[orders["status"] != "Cancelled"]
rfm = valid_orders.groupby("customer_id").agg(last_order=("created_at", "max"), frequency=("order_id", "count"),
                                              monetary=("subtotal", "sum"))
rfm["recency"] = (pd.Timestamp(CUTOFF) - rfm["last_order"]).dt.days
rfm["r_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.cut(rfm["frequency"], bins=[0, 1, 2, 3, 5, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m_score"] = pd.qcut(rfm["monetary"], 5, labels=[1, 2, 3, 4, 5]).astype(int)


def rfm_segment(r_score, f_score):
    """Return the RFM segment of a customer's recency and frequency scores."""
    if r_score >= 4 and f_score >= 4:
        return "Champions"
    if f_score >= 3:
        return "Loyal"
    if r_score >= 4 and f_score == 1:
        return "New"
    if r_score <= 2 and f_score >= 2:
        return "At risk"
    if r_score <= 2:
        return "Lost"
    return "Promising"


def row_segment(row):
    """Return the RFM segment of one row of the rfm table."""
    return rfm_segment(row["r_score"], row["f_score"])


rfm["segment"] = rfm.apply(row_segment, axis=1)
rfm["segment"].value_counts()

In [ ]:
two_channels = h1_customers[h1_customers["acquisition_channel"].isin(CHANNELS)]
two_channels = two_channels.merge(rfm[["segment"]], left_on="customer_id", right_index=True, how="inner", validate="one_to_one")
segment_mix = pd.crosstab(two_channels["acquisition_channel"], two_channels["segment"], normalize="index")
segment_mix.round(3), pd.crosstab(two_channels["acquisition_channel"], two_channels["segment"])

**Email and Paid Social customers, acquired January to June 2024.** 756 customers made their first purchase through Email and 757 through Paid Social.

- **Within a year, Email's customers came back almost twice as often.** 35.3% of them ordered again within 365 days of their first order, 267 customers, against 19.6% of Paid Social's, 148. Across all channels, it's 27.4%.
- **Today, Email's customers are worth more.** On January 1, 2026, 7.6% of Email's customers are Champions and 11.4% Loyal, against 0.8% and 4.2% of Paid Social's. 72.7% of Paid Social's customers are Lost, against 58.3% of Email's.
- **Part of the gap is who they are.** Email's customers had subscribed to the newsletter before they bought, so they were already interested in the store. Paid Social reached people who hadn't heard of it. This comparison can't say how much of the gap the channel itself causes.
- **Next step.** Compare the channels on what a customer brings in over a year, not on the cost of the first order: at the same cost per new customer, an Email customer is worth more. Send Email's 120 At risk customers a win-back offer now. For Paid Social, test whether a second-order offer in the first month lifts its repeat rate before cutting its budget.

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    def near(value, *targets, tolerance=0.0001):
        return number(value) and any(abs(value - target) < tolerance for target in targets)

    orders = found.get("orders")
    if not isinstance(orders, pd.DataFrame) or "created_at" not in orders.columns:
        results.append((False, "orders should be a DataFrame: ORDERS_FILE, with created_at as dates."))
    elif len(orders) == 65106:
        results.append((False, "The orders table has all 65,106 orders: 31 of them were placed on January 1, 2026 in UTC, after the data ends. Keep the orders before CUTOFF."))
    elif len(orders) != 65075:
        results.append((False, f"The orders table should have the 65,075 orders before CUTOFF. It has {len(orders):,}."))
    else:
        results.append((True, "65,075 orders before CUTOFF."))

    h1 = found.get("h1_customers")
    if not isinstance(h1, pd.DataFrame):
        results.append((False, "h1_customers should be a DataFrame: the customers created from COHORT_START to before COHORT_END."))
    elif len(h1) == 8801:
        results.append((False, "The h1_customers table has 8,801 customers: it starts each customer at their first order in the file, so 2023 customers who came back in 2024 count as new. Use the customers table's created_at."))
    elif len(h1) != 7545:
        results.append((False, f"The h1_customers table should have the 7,545 customers created from COHORT_START to before COHORT_END. It has {len(h1):,}."))
    else:
        results.append((True, "7,545 customers created from January to June 2024: 756 through Email and 757 through Paid Social."))

    rates = found.get("repeat_rates")
    if not isinstance(rates, pd.Series) or "Email" not in rates.index or "Paid Social" not in rates.index:
        results.append((False, "repeat_rates should be a Series with one share per acquisition_channel."))
    elif len(rates) != 9:
        results.append((False, f"repeat_rates should have the 9 acquisition channels. It has {len(rates)}."))
    elif near(rates["Email"], 1.0) and near(rates["Paid Social"], 1.0, 0.998679):
        results.append((False, "repeat_rates is close to 1 for every channel: it counts the first order as a repeat. Keep the orders after each customer's first order: the customer record is created a few minutes before it."))
    elif near(rates["Email"], 0.424603):
        results.append((False, "repeat_rates for Email is 0.4246: it counts every later order, with no window. Keep the orders within 365 days of the first one, so every customer has had the same time to come back."))
    elif near(rates["Email"], 0.335979):
        results.append((False, "repeat_rates for Email is 0.3360: it leaves out cancelled orders. The brief counts an order of any status, because a cancelled order still shows that the customer came back."))
    elif not near(rates["Email"], 0.353175) or not near(rates["Paid Social"], 0.195509):
        results.append((False, f"repeat_rates should be about 0.3532 for Email and 0.1955 for Paid Social. They are {rates['Email']!r} and {rates['Paid Social']!r}."))
    else:
        results.append((True, f"Within a year, {rates['Email']:.1%} of Email's customers ordered again, and {rates['Paid Social']:.1%} of Paid Social's: {rates['Email'] / rates['Paid Social']:.1f} times as many."))

    rfm = found.get("rfm")
    if not isinstance(rfm, pd.DataFrame) or "segment" not in rfm.columns:
        results.append((False, "rfm should be a DataFrame with a segment column: topic 11.3.'s table, for every customer with a valid order."))
    elif len(rfm) != 41776:
        results.append((False, f"The rfm table should have the 41,776 customers with a valid order before CUTOFF: score every customer, not only the cohort's. It has {len(rfm):,}."))
    elif rfm["segment"].value_counts().to_dict() != {"Lost": 14069, "Promising": 11437, "New": 10650, "At risk": 2101, "Loyal": 2071, "Champions": 1448}:
        results.append((False, "The rfm table's segments should match topic 11.3.'s exercise: 14,069 Lost customers and 1,448 Champions. Check the scores and the order of rfm_segment()'s rules."))
    else:
        results.append((True, "Every customer's RFM segment as of January 1, 2026, as in topic 11.3.'s exercise."))

    mix = found.get("segment_mix")
    if not isinstance(mix, pd.DataFrame) or set(mix.index) != {"Email", "Paid Social"}:
        results.append((False, "segment_mix should be a DataFrame with a row for Email and one for Paid Social."))
    elif "Champions" not in mix.columns or "Lost" not in mix.columns:
        results.append((False, "segment_mix should have one column per segment, such as Champions and Lost."))
    elif near(mix.loc["Email", "Champions"], 0.129076):
        results.append((False, "segment_mix shows 12.9% of Email's customers as Champions: the scores were calculated among the cohort's customers only. Score every customer first, then look at the cohort."))
    elif not near(mix.loc["Email", "Champions"], 0.076087) or not near(mix.loc["Paid Social", "Lost"], 0.727019):
        results.append((False, f"segment_mix should have the share of each channel's cohort customers in each segment, with each row adding up to 1: 7.6% of Email's are Champions, and 72.7% of Paid Social's are Lost. It has {mix.loc['Email', 'Champions']!r} and {mix.loc['Paid Social', 'Lost']!r}."))
    else:
        results.append((True, "7.6% of Email's customers are Champions and 11.4% Loyal, against 0.8% and 4.2% of Paid Social's. 72.7% of Paid Social's are Lost."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Read your text cell again: it's what the media team will budget with.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 1: the files</b></summary>

Topic 11.2.: read both files with `parse_dates`, and keep the orders before `CUTOFF`.

~~~python
customers = pd.read_csv(CUSTOMERS_FILE, parse_dates=[...])
orders = pd.read_csv(ORDERS_FILE, parse_dates=[...])
orders = orders[orders["created_at"] < ...]
~~~

</details>

<details>
<summary><b>Step 2: the cohort</b></summary>

Topic 11.2.: start from the customer record, not from the first order in the file, which for a 2023 customer isn't their first.

~~~python
h1_customers = customers[(customers["created_at"] >= ...) & (customers["created_at"] < ...)]
len(h1_customers)
~~~

</details>

<details>
<summary><b>Step 3: the repeat rates</b></summary>

Find each cohort customer's first order, keep the orders after it and within 365 days of it, and mark the customers who have one. The share of `True` per channel is a mean (topic 8.2.).

~~~python
h1_orders = orders[orders["customer_id"].isin(...)]
first_orders = h1_orders.groupby("customer_id")["created_at"].min().rename("first_order")
h1_orders = h1_orders.merge(first_orders, on="customer_id", validate="many_to_one")
again = h1_orders[(h1_orders["created_at"] > ...) & (h1_orders["created_at"] <= ... + pd.Timedelta(days=365))]
repeated = h1_customers["customer_id"].isin(...)
repeat_rates = repeated.groupby(h1_customers["acquisition_channel"]).mean()
~~~

</details>

<details>
<summary><b>Step 4: the RFM segments</b></summary>

Topic 11.3.'s exercise, on every customer: a score is relative to the customers you scored, so score them all before you look at the cohort.

~~~python
valid_orders = orders[orders["status"] != ...]
rfm = valid_orders.groupby("customer_id").agg(last_order=..., frequency=..., monetary=...)
rfm["recency"] = (pd.Timestamp(CUTOFF) - rfm["last_order"]).dt.days
rfm["r_score"] = pd.qcut(rfm["recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["f_score"] = pd.cut(rfm["frequency"], bins=[0, 1, 2, 3, 5, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)
rfm["m_score"] = ...


def rfm_segment(r_score, f_score):
    ...
~~~

</details>

<details>
<summary><b>Step 5: the segment mix</b></summary>

Join the two channels' cohort customers to their segments with an inner join, then topic 9.3.'s `pd.crosstab()`, with `normalize="index"` to make each row's shares add up to 1.

~~~python
two_channels = h1_customers[h1_customers["acquisition_channel"].isin(...)]
two_channels = two_channels.merge(rfm[["segment"]], left_on="customer_id", right_index=True, how=...)
segment_mix = pd.crosstab(two_channels[...], two_channels[...], normalize="index")
~~~

</details>

<details>
<summary><b>Step 6: the text cell</b></summary>

Choose **Insert → Text cell**. Start with the repeat rates, with the customer counts behind them. Then compare the segments that matter for revenue: Champions, Loyal and At risk. Before you recommend anything, say what else could explain the gap: Email's customers had subscribed to the newsletter before they bought. End with how the media team should compare the channels' costs.

</details>